In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
df = sns.load_dataset("titanic")
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
#Here we are only choosing only some features from the total titanic dataset for our model to get trained only on this

features = ["pclass", "sex", "fare", "embarked", "age"]
target = ["survived"]

In [ ]:
# missing data filling.

from sklearn.impute import SimpleImputer

imp_median = SimpleImputer(strategy="median")
df[["age"]] = imp_median.fit_transform(df[["age"]])

imp_freq = SimpleImputer(strategy="most_frequent")
df[["embarked"]] = imp_freq.fit_transform(df[["embarked"]])

In [ ]:
df.isnull().sum()

In [ ]:
# Encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sex"] = le.fit_transform(df["sex"])
df["embarked"] = le.fit_transform(df["embarked"])

In [ ]:
df.head()

In [ ]:
X = df[features]
y = df[target]

X.head()

In [ ]:
y.head()

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)


# Decision tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier()
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

print("accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
# Plotting decision tree

from sklearn.tree import plot_tree

plt.figure(figsize=(18,10))
plot_tree(
    model,
    feature_names=X.columns,
    class_names=["Died", "Survived"],
    filled=True,
    # max_depth=2
)
plt.tight_layout()
plt.show()

# Decision tree with Pre-Pruning

In [ ]:
max_depth = [2,3,4,5,6,7,8,9,10]

for i in max_depth:
    model = DecisionTreeClassifier(max_depth=i)
    model.fit(X_train, y_train)

    # we can directly calc accuracy using the inbuilt fnx called score
    acc = model.score(X_test, y_test)
    print(f"depth={i}, accuracy={acc}")

# Decision tree with Post-Pruning

In [ ]:
model = DecisionTreeClassifier()
model.fit(X_train, y_train)

In [ ]:
path = model.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

print(ccp_alphas)

In [ ]:
trees = []

for alpha in ccp_alphas:
    model = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    model.fit(X_train, y_train)

    trees.append((model, alpha))

In [ ]:
best_acc = 0
best_alpha = 0

for model, alpha in trees:
    curr_acc = model.score(X_test, y_test)
    if curr_acc > best_acc:
        best_acc = curr_acc
        best_alpha = alpha

In [ ]:
best_alpha

In [ ]:
best_acc

In [ ]:
best_model = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha)
best_model.fit(X_train, y_train)

In [ ]:
plt.figure(figsize=(18,10))
plot_tree(
    best_model,
    feature_names=X.columns,
    class_names=["Died", "Survived"],
    filled=True,
)
plt.tight_layout()
plt.show()

In [ ]:
print(best_model.score(X_test, y_test))

# Decision tree Regressor

In [ ]:
from sklearn.datasets import load_diabetes

In [ ]:
df = load_diabetes(as_frame=True).frame #shortest way out to create a df
df.head()

In [ ]:
X = df.drop(columns = ("target"))
y = df["target"]

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)


In [ ]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(max_depth=7, min_samples_leaf=20) # max_depth and min_samples_leaf is pre-pruning
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

# Here we are checking both training and testing results
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print("MSE train:", mean_squared_error(y_train, y_pred_train))
print("MSE test:", mean_squared_error(y_test, y_pred_test))

print("r2 train:", r2_score(y_train, y_pred_train))
print("r2 test:", r2_score(y_test, y_pred_test))

In [ ]:
# plotting DT regressor

from sklearn.tree import plot_tree

plt.figure(figsize=(18,10))

plot_tree(
    model,
    feature_names=X.columns,
    filled=True,
    # max_depth=1
)
plt.tight_layout()